In [ ]:
import pandas as pd
import numpy as np
import csv
import os
import plotly.express as px

import torch
import torch.nn as nn
import torchinfo
import torchvision
from tqdm import tqdm

from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.transforms import v2

from model.loader import load_dataset, load_test_loader
from utils.CustomDataset import CustomDataset
from utils.CustomSkibidiDataset import CustomSkibidiDataset

import torchvision.models as models
from eval.Analyser import Analyser, plot_f1_report
from result.repondeur import prediction_val_to_csv

In [ ]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU - training will be slow!")

In [ ]:
def train_crossEntropyLoss(train_set, val_set, model, epochs):
    """
    Entraine un model à partir d'un train_set donné. Retourne les metrics de l'entrainement
    """

    # Parameters
    model.train()
    
    criterion = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

    # optimizer = torch.optim.SGD(
    #     model.parameters(),
    #     lr=0.1,
    #     momentum=0.9,
    #     weight_decay=5e-4
    # )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=200
    )


    accuracies = np.zeros(epochs)
    losses = np.zeros(epochs)
    f1scores = np.zeros(epochs)

    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(
            train_set,
            desc=f"Model training | Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            #prediction
            logits = model(images)
            loss = criterion(logits, labels)

            #propagation du gradient
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Calcul des metriques
            total_loss += loss.item()
            
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                "batch_loss": f"{loss.item():.4f}",
                "accuracy": f"{correct/total*100:.2f}%"
            })
        accuracies[epoch] = correct/total
        losses[epoch] = total_loss

        scheduler.step()

        model_epoch_path =  f"model/trained_model/ResNet_augmented_data_{epoch + 1}_epochs.pth"
        torch.save(model.state_dict(), model_epoch_path)

        # Validation
        val_csv_path = f"model/trained_model/prediction_validation_{epoch + 1}_epochs.csv"
        prediction_val_to_csv(dataset=val_set, model=model, output_path=val_csv_path)

        analyser = Analyser(pd.read_csv(val_csv_path))
        report = analyser.generate_report()
        print(report)
        f1scores[epoch]=report['f1_score_avg']
    return accuracies, losses, f1scores

In [ ]:
# 1. Define Image Transforms
# HUUUUGOOOOOOOO
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     torchvision.transforms.functional.to_dtype(float16, scale=True),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

transform = v2.Compose([
    #v2.ToImage(),                  # Convert to Tensor (if it's PIL)
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True), # Scales to float16 [0, 1] # TODO
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# 2. Instantiate your Dataset
"""train_dataset = CustomDataset(
    img_dir='data/',
    annotations_file='data/train.csv',
    transform=transform, 
    target_transform=None # ???
)"""

skibidi_dataset = CustomSkibidiDataset(
    img_dir='data/treated_train/',
    csv_mapper='data/treated_train.csv',
    transform=transform, 
    target_transform=None # ???
)

train_size = int(0.8 * len(skibidi_dataset))
test_size = len(skibidi_dataset) - train_size

train_dataset, validation_dataset = random_split(skibidi_dataset, [train_size, test_size])

In [ ]:
if not(os.path.exists("model/trained_model/")):
    os.makedirs("model/trained_model/")

epochs = 20
model = models.resnet50(pretrained=True)
for param in model.parameters():
    param.requires_grad = False
num_classes = 50
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.to(device)

train_set = DataLoader(
    dataset=train_dataset,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

In [ ]:
accuracies, losses, f1scores = train_crossEntropyLoss(train_set, validation_dataset, model, epochs)

try : 
    np.save("./ACCURACIES.npy", accuracies)
    np.save("./LOSSES.npy", losses)
    np.save("./F1SCORES.npy", f1scores)
except :
    pass
torch.save(model.state_dict(), f"model/trained_model/ResNet_augmented_data_{epochs}_epochs.pth")

#### Analysis of model training

In [ ]:
# --- Load arrays ---
accuracies = np.load("./ACCURACIES.npy")
losses = np.load("./LOSSES.npy")
f1scores = np.load("./F1SCORES.npy")

# --- Create epoch axis ---
epochs = np.arange(len(accuracies))  # epoch 0 at index 0

# --- Put everything into a DataFrame ---
df = pd.DataFrame({
    "Epoch": epochs,
    "Accuracy": accuracies,
    "Loss": losses,
    "F1 Score": f1scores
})

# --- Convert to long format (best for plotly express) ---
df_long = df.melt(id_vars="Epoch",
                  value_vars=["Accuracy", "Loss", "F1 Score"],
                  var_name="Metric",
                  value_name="Value")

# --- Plot ---
fig = px.line(
    df_long,
    x="Epoch",
    y="Value",
    color="Metric",
    markers=True,
    title="Training Metrics vs Epochs"
)

fig.show()

Last epoch

In [ ]:
analyser = Analyser(pd.read_csv(f"model/trained_model/prediction_validation_{epochs}_epochs.csv"))
report = analyser.generate_report()
print(report)

In [ ]:
plot_f1_report(report)